[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/16_conditioning_and_dit.ipynb)

# 16. Conditioning and DiT modulation — FiLM, cross-attention, adaLN-Zero, and MMDiT

이 노트북은 condition을 neural network에 넣는 방식이 어떻게 발전하는지 비교한다.

이번 버전에서는 MMDiT를 단순한 `separate QKV + joint attention`으로 끝내지 않고, **각 modality가 독립적인 adaLN-style modulation과 residual gate를 가지는 구조**까지 포함한다.

모델 폭과 block 수는 작지만 DiT / SD3 계열의 핵심 계산 그래프는 유지한다.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)


## 1. FiLM: condition controls feature-wise scale and shift

FiLM은 condition에서 feature-wise affine parameter를 만든다.

`y = x * (1 + scale(c)) + shift(c)`


In [ ]:
features = torch.randn(
    2,
    5,
    8,
    device=device,
)

condition = torch.randn(
    2,
    4,
    device=device,
)

film_projection = nn.Linear(
    4,
    16,
).to(device)

scale, shift = film_projection(
    condition
).chunk(
    2,
    dim=-1,
)

film_output = (
    features
    * (1 + scale[:, None, :])
    + shift[:, None, :]
)

print("FiLM output:", film_output.shape)


## 2. Cross-attention

Latent diffusion 계열 cross-attention에서는 image/latent tokens가 query를 만들고 text condition tokens가 key/value를 만든다.

MMDiT처럼 두 modality가 같은 joint attention graph에 참여하는 것과는 구조가 다르다.


In [ ]:
image_tokens = torch.randn(
    2,
    6,
    12,
    device=device,
)

text_tokens = torch.randn(
    2,
    4,
    12,
    device=device,
)

query_projection = nn.Linear(
    12,
    12,
    bias=False,
).to(device)

key_projection = nn.Linear(
    12,
    12,
    bias=False,
).to(device)

value_projection = nn.Linear(
    12,
    12,
    bias=False,
).to(device)

q = query_projection(
    image_tokens
).view(
    2,
    6,
    3,
    4,
).transpose(
    1,
    2,
)

k = key_projection(
    text_tokens
).view(
    2,
    4,
    3,
    4,
).transpose(
    1,
    2,
)

v = value_projection(
    text_tokens
).view(
    2,
    4,
    3,
    4,
).transpose(
    1,
    2,
)

cross_attention = (
    F.scaled_dot_product_attention(
        q,
        k,
        v,
    )
)

print(
    "cross-attention:",
    cross_attention.shape,
)


## 3. DiT adaLN-Zero

DiT의 adaLN-Zero block은 attention branch와 MLP branch 각각에

- shift
- scale
- residual gate

를 만든다.

따라서 condition 하나에서 총 6개의 modulation vector가 나온다. Zero initialization을 사용하면 block은 초기에 identity에 가깝게 시작한다.


In [ ]:
def modulate(
    x,
    shift,
    scale,
):
    return (
        x
        * (1 + scale[:, None, :])
        + shift[:, None, :]
    )


class TinyAdaLNZeroBlock(nn.Module):
    def __init__(
        self,
        hidden_dim=12,
        num_heads=3,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )
        self.attention = nn.MultiheadAttention(
            hidden_dim,
            num_heads,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )
        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                4 * hidden_dim,
            ),
            nn.GELU(
                approximate="tanh"
            ),
            nn.Linear(
                4 * hidden_dim,
                hidden_dim,
            ),
        )

        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                hidden_dim,
                6 * hidden_dim,
            ),
        )

        nn.init.zeros_(
            self.modulation[-1].weight
        )
        nn.init.zeros_(
            self.modulation[-1].bias
        )

    def forward(
        self,
        x,
        condition,
    ):
        (
            shift_attn,
            scale_attn,
            gate_attn,
            shift_mlp,
            scale_mlp,
            gate_mlp,
        ) = self.modulation(
            condition
        ).chunk(
            6,
            dim=-1,
        )

        attention_input = modulate(
            self.norm1(x),
            shift_attn,
            scale_attn,
        )

        attention_output, _ = (
            self.attention(
                attention_input,
                attention_input,
                attention_input,
                need_weights=False,
            )
        )

        x = (
            x
            + gate_attn[:, None, :]
            * attention_output
        )

        mlp_input = modulate(
            self.norm2(x),
            shift_mlp,
            scale_mlp,
        )

        x = (
            x
            + gate_mlp[:, None, :]
            * self.mlp(mlp_input)
        )

        return x


dit_block = TinyAdaLNZeroBlock().to(
    device
)

dit_condition = torch.randn(
    2,
    12,
    device=device,
)

dit_output = dit_block(
    image_tokens,
    dit_condition,
)

print(
    "initial max residual change:",
    (
        dit_output
        - image_tokens
    ).abs().max().item(),
)


## 4. MMDiT requires separate modality parameters and joint attention

Stable Diffusion 3의 MMDiT에서는 image와 text stream이 같은 parameter를 공유하는 하나의 Transformer가 아니다.

각 modality는 자기

- LayerNorm
- condition modulation
- QKV projection
- output projection
- MLP

을 가진다.

다만 attention 계산에서는 image와 text에서 만든 Q/K/V를 token dimension으로 합쳐 **joint attention**을 수행한다. 이 때문에 image가 text를 읽을 뿐 아니라 text도 image를 읽을 수 있다.


In [ ]:
class ModalityParameters(nn.Module):
    def __init__(
        self,
        hidden_dim,
        num_heads,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = (
            hidden_dim // num_heads
        )

        self.norm1 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )
        self.norm2 = nn.LayerNorm(
            hidden_dim,
            elementwise_affine=False,
        )

        self.qkv = nn.Linear(
            hidden_dim,
            3 * hidden_dim,
        )
        self.out = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                4 * hidden_dim,
            ),
            nn.GELU(
                approximate="tanh"
            ),
            nn.Linear(
                4 * hidden_dim,
                hidden_dim,
            ),
        )

        self.modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                hidden_dim,
                6 * hidden_dim,
            ),
        )

        nn.init.zeros_(
            self.modulation[-1].weight
        )
        nn.init.zeros_(
            self.modulation[-1].bias
        )

    def modulation_vectors(
        self,
        condition,
    ):
        return self.modulation(
            condition
        ).chunk(
            6,
            dim=-1,
        )

    def project_qkv(
        self,
        x,
        shift,
        scale,
    ):
        normalized = modulate(
            self.norm1(x),
            shift,
            scale,
        )

        batch_size = x.size(0)
        sequence_length = x.size(1)

        qkv = self.qkv(normalized)

        qkv = qkv.view(
            batch_size,
            sequence_length,
            3,
            self.num_heads,
            self.head_dim,
        ).permute(
            2,
            0,
            3,
            1,
            4,
        )

        return qkv.unbind(0)


class TinyMMDiTBlock(nn.Module):
    def __init__(
        self,
        hidden_dim=12,
        num_heads=3,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.image = ModalityParameters(
            hidden_dim,
            num_heads,
        )
        self.text = ModalityParameters(
            hidden_dim,
            num_heads,
        )

    @staticmethod
    def merge_heads(x):
        return x.transpose(
            1,
            2,
        ).contiguous().flatten(
            2,
        )

    def forward(
        self,
        image_tokens,
        text_tokens,
        condition,
    ):
        (
            image_shift_attn,
            image_scale_attn,
            image_gate_attn,
            image_shift_mlp,
            image_scale_mlp,
            image_gate_mlp,
        ) = self.image.modulation_vectors(
            condition
        )

        (
            text_shift_attn,
            text_scale_attn,
            text_gate_attn,
            text_shift_mlp,
            text_scale_mlp,
            text_gate_mlp,
        ) = self.text.modulation_vectors(
            condition
        )

        image_q, image_k, image_v = (
            self.image.project_qkv(
                image_tokens,
                image_shift_attn,
                image_scale_attn,
            )
        )

        text_q, text_k, text_v = (
            self.text.project_qkv(
                text_tokens,
                text_shift_attn,
                text_scale_attn,
            )
        )

        joint_q = torch.cat(
            [image_q, text_q],
            dim=2,
        )
        joint_k = torch.cat(
            [image_k, text_k],
            dim=2,
        )
        joint_v = torch.cat(
            [image_v, text_v],
            dim=2,
        )

        joint_output = (
            F.scaled_dot_product_attention(
                joint_q,
                joint_k,
                joint_v,
            )
        )

        image_length = image_tokens.size(1)

        image_attention = joint_output[
            :,
            :,
            :image_length,
            :,
        ]
        text_attention = joint_output[
            :,
            :,
            image_length:,
            :,
        ]

        image_tokens = (
            image_tokens
            + image_gate_attn[:, None, :]
            * self.image.out(
                self.merge_heads(
                    image_attention
                )
            )
        )

        text_tokens = (
            text_tokens
            + text_gate_attn[:, None, :]
            * self.text.out(
                self.merge_heads(
                    text_attention
                )
            )
        )

        image_mlp_input = modulate(
            self.image.norm2(
                image_tokens
            ),
            image_shift_mlp,
            image_scale_mlp,
        )

        text_mlp_input = modulate(
            self.text.norm2(
                text_tokens
            ),
            text_shift_mlp,
            text_scale_mlp,
        )

        image_tokens = (
            image_tokens
            + image_gate_mlp[:, None, :]
            * self.image.mlp(
                image_mlp_input
            )
        )

        text_tokens = (
            text_tokens
            + text_gate_mlp[:, None, :]
            * self.text.mlp(
                text_mlp_input
            )
        )

        return image_tokens, text_tokens


mmdit = TinyMMDiTBlock().to(device)

mmdit_condition = torch.randn(
    2,
    12,
    device=device,
)

image_output, text_output = mmdit(
    image_tokens,
    text_tokens,
    mmdit_condition,
)

print(
    "image output:",
    image_output.shape,
)
print(
    "text output:",
    text_output.shape,
)
print(
    "joint token count:",
    image_tokens.size(1)
    + text_tokens.size(1),
)


## 5. Make the conditioning source explicit

실제 diffusion/rectified-flow transformer에서는 condition vector가 임의의 vector가 아니라 timestep embedding과 pooled text condition 등을 합쳐 만들어진다.

아래는 작은 time embedding과 pooled text embedding을 합쳐 MMDiT condition을 만드는 예다.


In [ ]:
class TinyConditionEmbedder(nn.Module):
    def __init__(
        self,
        hidden_dim=12,
    ):
        super().__init__()

        self.time_mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
            nn.SiLU(),
            nn.Linear(
                hidden_dim,
                hidden_dim,
            ),
        )

        self.text_pool_projection = nn.Linear(
            hidden_dim,
            hidden_dim,
        )

    def forward(
        self,
        time_embedding,
        pooled_text,
    ):
        return (
            self.time_mlp(
                time_embedding
            )
            + self.text_pool_projection(
                pooled_text
            )
        )


condition_embedder = (
    TinyConditionEmbedder().to(device)
)

time_embedding = torch.randn(
    2,
    12,
    device=device,
)
pooled_text = text_tokens.mean(
    dim=1
)

combined_condition = condition_embedder(
    time_embedding,
    pooled_text,
)

print(
    "combined condition:",
    combined_condition.shape,
)


## 6. Gradient-flow sanity check

Zero-initialized modulation layer에서는 첫 backward에서 QKV/MLP branch보다 modulation gate 쪽에 먼저 gradient가 생기는 것이 정상이다.

한 optimizer step을 수행한 뒤 다시 backward하여 joint-attention parameter까지 gradient가 열리는지 확인한다.


In [ ]:
train_mmdit = TinyMMDiTBlock().to(
    device
)

optimizer = torch.optim.SGD(
    train_mmdit.parameters(),
    lr=0.1,
)

condition = torch.randn(
    2,
    12,
    device=device,
)

for step in range(2):
    optimizer.zero_grad()

    image_out, text_out = train_mmdit(
        image_tokens,
        text_tokens,
        condition,
    )

    loss = (
        image_out.square().mean()
        + text_out.square().mean()
    )

    loss.backward()

    image_qkv_grad = (
        train_mmdit.image.qkv.weight.grad
    )
    image_mod_grad = (
        train_mmdit.image.modulation[
            -1
        ].weight.grad
    )

    print("step:", step)
    print(
        "modulation grad:",
        image_mod_grad.norm().item(),
    )
    print(
        "QKV grad:",
        0.0
        if image_qkv_grad is None
        else image_qkv_grad.norm().item(),
    )

    optimizer.step()


## References and provenance

**FiLM** — condition-dependent feature-wise affine modulation.

**DiT** — attention branch와 MLP branch 각각의 shift, scale, gate를 만드는 adaLN-Zero 구조와 zero initialization을 반영했다.

**Stable Diffusion 3 / MMDiT** — image와 text가 separate weights를 가지면서 Q/K/V를 합쳐 joint attention을 수행하는 구조를 반영했다. 이번 버전에서는 두 modality 각각에 condition-dependent modulation과 residual gate도 포함한다.

실제 SD3의 모델 폭, block 수, text encoders, VAE와 training schedule은 생략하지만 MMDiT block의 핵심 parameter topology는 유지한다.
